In [2]:
import typing, importlib, numpy as np, jax, jax.numpy as jnp, matplotlib.pyplot as plt, optax, tqdm

In [3]:
import metanet, simulationMetanet
importlib.reload(metanet)
importlib.reload(simulationMetanet)

<module 'simulationMetanet' from '/home/pesim/uni/thesis/simulationMetanet.py'>

In [4]:
traj_true, p_true, boundaries, init_stat = simulationMetanet.simulate_example()

In [33]:
from math import tau


class LearningParams(typing.NamedTuple):
    metanet: metanet.NetworkParameters
    log_vars: jax.Array

# ==========================================
# 1. LOSS FUNCTION: NEGATIVE LOG-LIKELIHOOD
# ==========================================
def nll_loss(params: LearningParams, traj_true: metanet.SimulationTrajectory, initial_state: metanet.NetworkState, boundaries: metanet.BoundarySequence):
    """
    Calculates the Negative Log-Likelihood loss.
    
    params: LearningParams containing METANET parameters and error variances
    traj_true: Ground truth trajectory data
    initial_state: Initial state of the network
    boundaries: Boundary conditions for the simulation
    """
    model_params = params.metanet
    log_vars = params.log_vars 
    
    # 1. Simulate trajectories using the current METANET parameters
    traj_pred = metanet.rollout_simulation(initial_state, boundaries, model_params)
    
    # 2. Compute errors between predicted and true trajectories
    error_rho = traj_pred.density - traj_true.density
    error_v   = traj_pred.speed - traj_true.speed
    
    # 3. Calculate Negative Log-Likelihood assuming Gaussian errors
    # Using log_vars to ensure strictly positive variance (var = exp(log_var))
    var_rho = jnp.exp(log_vars[0])
    var_v   = jnp.exp(log_vars[1])
    
    n = traj_true.density.size  # Total number of data points
    
    # NLL = (n/2)*log(var) + sum(error^2)/(2*var)
    loss_rho = 0.5 * n * log_vars[0] + 0.5 * jnp.sum(error_rho**2) / var_rho
    loss_v   = 0.5 * n * log_vars[1] + 0.5 * jnp.sum(error_v**2) / var_v
    
    total_loss = loss_rho + loss_v
    
    return total_loss

# ==========================================
# 2. OPTIMIZATION LOOP SETUP
# ==========================================
def run_identification():
    # Example dimensions from user setup
    N = 20
    K = 360 * 2 + 180  # 900 steps
    T_step = 10.0 / 3600.0  # 10 seconds in hours
    
    # Generate ground truth data using the actual simulation module
    print("Generating ground truth simulation data...")
    traj_true, p_true, boundaries, init_stat = simulationMetanet.simulate_example()
    
    # Define initial parameters
    # We initialize learnable parameters slightly off from their true values (e.g. * 0.8 or * 1.2)
    # and we perfectly initialize the non-learnable (frozen) parameters to their true ground values.
    dipersion = 0.05  # 20% deviation for initialization
    initial_metanet_params = metanet.NetworkParameters(
        tau=jax.random.uniform(jax.random.PRNGKey(0), (p_true.tau.shape), minval=(1-dipersion)*p_true.tau, maxval=(1+dipersion)*p_true.tau),         # Learnable (initialized off)
        free_flow_speed=jax.random.uniform(jax.random.PRNGKey(1), (p_true.free_flow_speed.shape), minval=(1-dipersion)*p_true.free_flow_speed, maxval=(1+dipersion)*p_true.free_flow_speed),   # Learnable (initialized off)
        kappa= jax.random.uniform(jax.random.PRNGKey(2), (p_true.kappa.shape), minval=(1-dipersion)*p_true.free_flow_speed, maxval=(1+dipersion)*p_true.kappa),     # Learnable (initialized off)
        nu=jax.random.uniform(jax.random.PRNGKey(3), (p_true.nu.shape), minval=(1-dipersion)*p_true.nu, maxval=(1+dipersion)*p_true.nu), # Learnable (initialized off)
        critical_density=jax.random.uniform(jax.random.PRNGKey(4), (p_true.critical_density.shape), minval=(1-dipersion)*p_true.critical_density, maxval=(1+dipersion)*p_true.critical_density),# Learnable (initialized off)
        alpha=jax.random.uniform(jax.random.PRNGKey(5), (p_true.alpha.shape), minval=(1-dipersion)*p_true.alpha, maxval=(1+dipersion)*p_true.alpha),             # Learnable (initialized off)
        delta=jax.random.uniform(jax.random.PRNGKey(6), (p_true.delta.shape), minval=(1-dipersion)*p_true.delta, maxval=(1+dipersion)*p_true.delta),     # Learnable (initialized off)
        L=p_true.L,                   # FROZEN
        lambda_=p_true.lambda_,               # FROZEN
        T=p_true.T                    # FROZEN
    )

    """
    initial_metanet_params = metanet.NetworkParameters(
        tau=p_true.tau,         # Learnable (initialized off)
        free_flow_speed=p_true.free_flow_speed,   # Learnable (initialized off)
        kappa= p_true.kappa,     # Learnable (initialized off)
        nu=p_true.nu, # Learnable (initialized off)
        critical_density=p_true.critical_density,# Learnable (initialized off)
        alpha=p_true.alpha,             # Learnable (initialized off)
        delta=p_true.delta,     # Learnable (initialized off)
        L=p_true.L,                   # FROZEN
        lambda_=p_true.lambda_,               # FROZEN
        T=p_true.T                    # FROZEN
    )
    """
    frac = jnp.sum(traj_true.flow) / jnp.sum(traj_true.speed)
    initial_params = LearningParams(
        metanet=initial_metanet_params,
        log_vars=jnp.array([jnp.log(0.5), jnp.log(10*frac)]) ) # Initialize log_vars at 0.0 (variance of 1.0)
    
    mask = LearningParams(
            metanet=metanet.NetworkParameters(
            tau=jnp.array(True),
            free_flow_speed=jnp.array(True),
            kappa=jnp.array(True),
            nu=jnp.array(True),
            critical_density=jnp.array(True),
            alpha=jnp.array(True),
            delta=jnp.array(True),
            L=jnp.array(False),   # FROZEN
            lambda_=jnp.array(False), # FROZEN
            T=0    # FROZEN
        ),
        log_vars=jnp.array(True) # Trainable
    )
    
    # Setup Optax Optimizer using optax.masked
    # This automatically applies updates ONLY where the mask is True.
    learning_rate = 1e-3
    optimizer = optax.masked(optax.adam(learning_rate), mask)
    
    opt_state = optimizer.init(initial_params)
    
    # Define a JIT-compiled update step for speed
    @jax.jit
    def update_step(params, opt_state, traj_true, initial_state, boundaries):
        # value_and_grad computes both the loss and the gradients of the PyTree
        loss, grads = jax.value_and_grad(nll_loss)(params, traj_true, initial_state, boundaries)
        
        # Calculate parameter updates based on gradients
        updates, opt_state = optimizer.update(grads, opt_state, params)
        
        # Apply updates to parameters
        new_params = optax.apply_updates(params, updates)
        
        return new_params, opt_state, loss

    # ==========================================
    # 3. TRAINING LOOP
    # ==========================================
    num_epochs = 500
    params = initial_params
    
    print("\nStarting parameter identification...")
    pbar = tqdm.tqdm(range(num_epochs))
    for epoch in pbar:
        params, opt_state, loss = update_step(params, opt_state, traj_true, init_stat, boundaries)
        
        # Update progress bar every 50 epochs
        if epoch % 50 == 0:
            var_rho = jnp.exp(params.log_vars[0])
            var_v = jnp.exp(params.log_vars[1])
            pbar.set_description(f"Loss: {loss:.4f} | Var Rho: {var_rho:.3f} | Var V: {var_v:.3f}")

    print("\nIdentified METANET Parameters vs True Parameters:")
    
    # Pretty print the comparison
    for field in params.metanet._fields:
        learned_val = getattr(params.metanet, field)
        true_val = getattr(p_true, field)
        frozen_str = "(FROZEN)" if field in ['L', 'lam', 'T'] else ""
        
        # Handle scalar vs array formatting safely
        if jnp.size(learned_val) == 1:
            l_str = f"{float(jnp.squeeze(learned_val)):.4f}"
            t_str = f"{float(jnp.squeeze(true_val)):.4f}"
        else:
            # Convert to numpy and limit printing size for cleaner output
            with np.printoptions(precision=4, suppress=True, edgeitems=2, threshold=5):
                l_str = f"{np.array(learned_val)}"
                t_str = f"{np.array(true_val)}"
            
        print(f"  {field:>10}: {l_str} \t(True: {t_str}) {frozen_str}")
        
    print("\nLearned Error Covariances (Variances):")
    print(f"  Density (rho) variance : {float(jnp.exp(params.log_vars[0])):.4f}")
    print(f"  Speed (v) variance     : {float(jnp.exp(params.log_vars[1])):.4f}")

run_identification()


Generating ground truth simulation data...

Starting parameter identification...


Loss: nan | Var Rho: nan | Var V: nan:  12%|█▏        | 62/500 [00:12<01:31,  4.81it/s]             


KeyboardInterrupt: 